<a href="https://colab.research.google.com/github/talaA2/IT362-DataScienceProject/blob/main/Phase%203/Notebooks/Baseline_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Baseline Model – Logistic Regression
TF-IDF features + Logistic Regression with class weighting, as a reference point for the other models.

In [1]:
import pandas as pd
from pathlib import Path

# Works from the repository (Phase 3/Notebooks) and in Colab (reads the copy on GitHub)
DATA_URL = "https://raw.githubusercontent.com/talaA2/IT362-DataScienceProject/main/Phase%203/labeled%20data%20files/"
def data_path(name):
    for folder in [Path("../labeled data files"), Path("Phase 3/labeled data files"), Path(".")]:
        if (folder / name).exists():
            return folder / name
    return DATA_URL + name

youtube = pd.read_csv(data_path("youtube_all_labeled.csv"))
gnews = pd.read_csv(data_path("Gnews_all_labeled.csv"))

print("YouTube shape:", youtube.shape)
print("GNews shape:", gnews.shape)

YouTube shape: (3781, 2)
GNews shape: (110, 2)


In [2]:
# Same combined dataset in every Phase 3 notebook: YouTube first, then GNews
youtube["source"] = "youtube"
gnews["source"] = "gnews"
df = pd.concat([youtube, gnews], ignore_index=True)

df = df.dropna(subset=["text_clean", "label"]).copy()
df["text_clean"] = df["text_clean"].astype(str)
df["label"] = df["label"].astype(int)

print("Final shape:", df.shape)
print("\nLabel distribution (-1 = negative, 0 = neutral, 1 = positive):")
print(df["label"].value_counts())
print("\nSource distribution:")
print(df["source"].value_counts())

Final shape: (3891, 3)

Label distribution (-1 = negative, 0 = neutral, 1 = positive):
label
-1    2260
 0     839
 1     792
Name: count, dtype: int64

Source distribution:
source
youtube    3781
gnews       110
Name: count, dtype: int64


In [3]:
from sklearn.model_selection import train_test_split

X = df["text_clean"]
y = df["label"]

# Same 80/20 split in every notebook; stratify keeps the class proportions equal in both parts
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training size:", X_train.shape[0])
print("Testing size:", X_test.shape[0])

Training size: 3112
Testing size: 779


In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, f1_score

# Convert text to numerical features
vectorizer = TfidfVectorizer(stop_words="english", max_features=5000)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

# Train Logistic Regression model (class_weight handles the class imbalance)
model = LogisticRegression(max_iter=1000, class_weight="balanced")
model.fit(X_train_vec, y_train)

# Evaluate model
y_pred = model.predict(X_test_vec)
print("Macro F1:", round(f1_score(y_test, y_pred, average="macro"), 4))
print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred, target_names=["negative", "neutral", "positive"]))

Macro F1: 0.6367
Accuracy: 0.6752

Classification Report:

              precision    recall  f1-score   support

    negative       0.81      0.71      0.76       452
     neutral       0.50      0.71      0.59       168
    positive       0.58      0.55      0.56       159

    accuracy                           0.68       779
   macro avg       0.63      0.65      0.64       779
weighted avg       0.70      0.68      0.68       779

